In [1]:
# Cell 1 — dependency check / (light) install
import importlib
import sys
import subprocess

def try_import(pkg):
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "unknown")
        print(f"✅ {pkg} imported (version: {ver})")
        return True
    except Exception as e:
        print(f"❌ {pkg} not available: {e}")
        return False

have_numpy = try_import("numpy")
have_torch = try_import("torch")
have_pl    = try_import("pennylane")

# Pennylane is usually light enough to pip install in a notebook; torch is not.
if not have_pl:
    print("Installing pennylane (lightweight)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    have_pl = try_import("pennylane")

if not have_torch:
    print("\n⚠️ PyTorch not found.")
    print("   - If you're on AWS Braket notebooks, torch is often preinstalled.")
    print("   - If not, you can either (a) pip install torch (can be large), or")
    print("     (b) use the fallback 'linear encoder' path later in this notebook.")

✅ numpy imported (version: 2.2.6)
❌ torch not available: No module named 'torch'
✅ pennylane imported (version: 0.44.1)

⚠️ PyTorch not found.
   - If you're on AWS Braket notebooks, torch is often preinstalled.
   - If not, you can either (a) pip install torch (can be large), or
     (b) use the fallback 'linear encoder' path later in this notebook.


In [3]:
# Cell 2 — imports
import numpy as np

# Optional: torch for the classical encoder (recommended)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except Exception:
    TORCH_OK = False

import pennylane as qml

In [19]:
def generate_regime_data(n_samples, seed=42):
    """
    Generate synthetic regime-switching data per the challenge spec.
    
    Returns:
        X: array of shape (n_samples, 4) — features X1, X2, X3, X4
        y: array of shape (n_samples,) — target variable
        regimes: array of shape (n_samples,) — 0 for Regime 1, 1 for Regime 2
    """
    rng = np.random.default_rng(seed)
    
    # Assign regimes: 75% Regime 1, 25% Regime 2
    regimes = rng.choice([0, 1], size=n_samples, p=[0.75, 0.25])
    
    n1 = np.sum(regimes == 0)
    n2 = np.sum(regimes == 1)
    
    X = np.zeros((n_samples, 4))
    y = np.zeros(n_samples)
    
    # --- Regime 1 ---
    mask1 = regimes == 0
    X[mask1, 0] = rng.standard_normal(n1)          # X1 ~ N(0,1)
    X[mask1, 1] = rng.standard_normal(n1)          # X2 ~ N(0,1)
    X[mask1, 2] = rng.standard_normal(n1)          # X3 ~ N(0,1)
    X[mask1, 3] = rng.uniform(-1, 1, size=n1)     # X4 ~ Uniform(-1,1)
    eps1 = rng.standard_normal(n1)                  # noise
    
    y[mask1] = 2 * X[mask1, 0] - X[mask1, 1] + eps1
    
    # --- Regime 2 ---
    mask2 = regimes == 1
    
    # X1 and X3 are jointly normal with correlation 0.8
    mean = [3, 3]
    cov = [[1, 0.8],
           [0.8, 1]]
    x1_x3 = rng.multivariate_normal(mean, cov, size=n2)
    X[mask2, 0] = x1_x3[:, 0]                      # X1
    X[mask2, 2] = x1_x3[:, 1]                      # X3
    
    # X2 ~ Cauchy(0,1)
    X[mask2, 1] = rng.standard_cauchy(size=n2)
    
    # X4 ~ Exponential(lambda=1)
    X[mask2, 3] = rng.exponential(scale=1.0, size=n2)
    
    eps2 = rng.standard_normal(n2)                  # noise
    
    y[mask2] = (X[mask2, 0] * X[mask2, 2]) + np.log(np.abs(X[mask2, 1]) + 1) + eps2
    
    return X, y, regimes

# EXACT split requested
X_train, y_train, regimes_train = generate_regime_data(10000, seed=42)
X_test, y_test, regimes_test = generate_regime_data(10000, seed=123)

# Printouts requested
print("\nTrain/Test shapes and regime counts")
print("X_train:", X_train.shape, "y_train:", y_train.shape, "regimes_train:", regimes_train.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape, "regimes_test :", regimes_test.shape)
print("Train regime counts:", np.bincount(regimes_train))
print("Test regime counts :", np.bincount(regimes_test))

print("\nTrain/Test y mean/std")
print("Train y mean/std:", float(np.mean(y_train)), float(np.std(y_train)))
print("Test  y mean/std:", float(np.mean(y_test)), float(np.std(y_test)))

print("\nRegime 2 X2 min/max")
train_r2 = X_train[regimes_train == 1, 1]
test_r2 = X_test[regimes_test == 1, 1]
print("Train Regime 2 X2 min/max:", float(np.min(train_r2)), float(np.max(train_r2)))
print("Test  Regime 2 X2 min/max:", float(np.min(test_r2)), float(np.max(test_r2)))



Train/Test shapes and regime counts
X_train: (10000, 4) y_train: (10000,) regimes_train: (10000,)
X_test : (10000, 4) y_test : (10000,) regimes_test : (10000,)
Train regime counts: [7558 2442]
Test regime counts : [7633 2367]

Train/Test y mean/std
Train y mean/std: 2.632977148389309 5.947593604110127
Test  y mean/std: 2.5971383479190853 5.83952693690379

Regime 2 X2 min/max
Train Regime 2 X2 min/max: -614.5850507495128 4488.181692438623
Test  Regime 2 X2 min/max: -1158.4316564492472 1350.6850911215495


In [20]:
# Cell 4 — standardize features (train stats only)

def standardize_fit(X):
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return mu, sd

def standardize_apply(X, mu, sd):
    return (X - mu) / sd

X_mu, X_sd = standardize_fit(X_train)
X_train_s = standardize_apply(X_train, X_mu, X_sd)
X_test_s  = standardize_apply(X_test,  X_mu, X_sd)

# (Optional) standardize y as well — often helps training the encoder
y_mu = y_train.mean()
y_sd = y_train.std() if y_train.std() > 1e-12 else 1.0
y_train_s = (y_train - y_mu) / y_sd
y_test_s  = (y_test  - y_mu) / y_sd

print("X_train_s mean (approx):", X_train_s.mean(axis=0))
print("X_train_s std  (approx):", X_train_s.std(axis=0))

X_train_s mean (approx): [-8.45870596e-16  3.52703977e-18 -2.18038088e-15  9.80510118e-16]
X_train_s std  (approx): [1. 1. 1. 1.]


In [21]:
# Cell 5 — Ridge regression (closed form), no sklearn

def ridge_fit_closed_form(X, y, lam=1.0, fit_intercept=True):
    """
    Solve (X^T X + lam*I) w = X^T y.
    If fit_intercept=True, we augment with a column of ones and do NOT penalize the intercept.
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1, 1)
    n, d = X.shape

    if fit_intercept:
        X_aug = np.concatenate([np.ones((n, 1)), X], axis=1)
        d_aug = d + 1
        A = X_aug.T @ X_aug
        reg = np.eye(d_aug)
        reg[0, 0] = 0.0  # don't penalize intercept
        A = A + lam * reg
        b = X_aug.T @ y
        w = np.linalg.solve(A, b).reshape(-1)
        return w  # w[0] intercept, w[1:] weights
    else:
        A = X.T @ X + lam * np.eye(d)
        b = X.T @ y
        w = np.linalg.solve(A, b).reshape(-1)
        return w

def ridge_predict(X, w, fit_intercept=True):
    X = np.asarray(X, dtype=np.float64)
    if fit_intercept:
        return (w[0] + X @ w[1:]).reshape(-1)
    return (X @ w).reshape(-1)

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if y_true.size > 1 else float("nan")
    return {"MSE": mse, "MAE": mae, "Corr": corr}

LAMBDA = 1.0  # ridge penalty; keep fixed for apples-to-apples

# Baseline: ridge on standardized raw X
w_raw = ridge_fit_closed_form(X_train_s, y_train, lam=LAMBDA, fit_intercept=True)
pred_raw = ridge_predict(X_test_s, w_raw, fit_intercept=True)
print("Baseline Ridge (raw X):", regression_metrics(y_test, pred_raw))

Baseline Ridge (raw X): {'MSE': 4.5710936347281415, 'MAE': 1.6204180219195057, 'Corr': 0.9305954635118744}


In [22]:
# Cell 6 — Train a supervised classical encoder (PyTorch) to get latent z
# If torch is unavailable, we fall back to a simple linear "encoder" (identity / PCA-like).

LATENT_DIM = 4  # number of qubits later; with 4 base features, 4 is natural. You can set 6 too.
HIDDEN = 32
EPOCHS = 15
BATCH = 256
LR = 1e-3

if TORCH_OK:
    class EncoderNet(nn.Module):
        def __init__(self, in_dim, hidden, latent_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, latent_dim),
                nn.Tanh(),  # bound to [-1, 1] for stable quantum angle scaling
            )

        def forward(self, x):
            return self.net(x)

    class EncoderPlusHead(nn.Module):
        def __init__(self, in_dim, hidden, latent_dim):
            super().__init__()
            self.encoder = EncoderNet(in_dim, hidden, latent_dim)
            self.head = nn.Linear(latent_dim, 1)

        def forward(self, x):
            z = self.encoder(x)
            yhat = self.head(z)
            return z, yhat

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EncoderPlusHead(in_dim=X_train_s.shape[1], hidden=HIDDEN, latent_dim=LATENT_DIM).to(device)

    Xtr_t = torch.tensor(X_train_s, dtype=torch.float32)
    ytr_t = torch.tensor(y_train_s.reshape(-1, 1), dtype=torch.float32)
    ds = TensorDataset(Xtr_t, ytr_t)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=False)

    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    model.train()
    for ep in range(1, EPOCHS + 1):
        running = 0.0
        for xb, yb in dl:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            _, yhat = model(xb)
            loss = loss_fn(yhat, yb)
            loss.backward()
            opt.step()
            running += float(loss.item()) * xb.size(0)
        print(f"Epoch {ep:02d} | train MSE (standardized y): {running / len(ds):.6f}")

    # Freeze encoder and extract latent z for train/test
    model.eval()
    with torch.no_grad():
        Z_train = model.encoder(torch.tensor(X_train_s, dtype=torch.float32).to(device)).cpu().numpy()
        Z_test  = model.encoder(torch.tensor(X_test_s,  dtype=torch.float32).to(device)).cpu().numpy()

else:
    print("TORCH not available — using fallback linear 'encoder' (identity/truncate).")
    # If LATENT_DIM > input dim, pad with zeros.
    def linear_encoder(X, latent_dim):
        X = np.asarray(X, dtype=np.float64)
        if latent_dim <= X.shape[1]:
            return X[:, :latent_dim]
        pad = np.zeros((X.shape[0], latent_dim - X.shape[1]))
        return np.concatenate([X, pad], axis=1)

    Z_train = linear_encoder(X_train_s, LATENT_DIM)
    Z_test  = linear_encoder(X_test_s,  LATENT_DIM)

print("Latent shapes:", Z_train.shape, Z_test.shape)

# Compressed-feature baseline: ridge on Z (same model class)
w_z = ridge_fit_closed_form(Z_train, y_train, lam=LAMBDA, fit_intercept=True)
pred_z = ridge_predict(Z_test, w_z, fit_intercept=True)
print("Baseline Ridge (compressed Z):", regression_metrics(y_test, pred_z))

TORCH not available — using fallback linear 'encoder' (identity/truncate).
Latent shapes: (10000, 4) (10000, 4)
Baseline Ridge (compressed Z): {'MSE': 4.5710936347281415, 'MAE': 1.6204180219195057, 'Corr': 0.9305954635118744}


In [23]:
# Cell 7 — Quantum feature map + feature extraction

N_QUBITS = LATENT_DIM
FEATURE_MAP = "zz"   # "zz" | "iqp" | "reupload"
SHOTS = None         # None => analytic expectations (fast & differentiable); set int for sampling realism

qdev = qml.device("default.qubit", wires=N_QUBITS, shots=SHOTS)

def _entangle_chain():
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i+1])

def _zz_feature_map(z):
    # Data-dependent single-qubit phases + pairwise ZZ couplings
    for i in range(N_QUBITS):
        qml.RZ(z[i], wires=i)
    for i in range(N_QUBITS - 1):
        # IsingZZ(phi) = exp(-i phi/2 Z⊗Z)
        qml.IsingZZ(z[i] * z[i+1], wires=[i, i+1])

def _iqp_feature_map(z):
    # IQP-style: H -> diagonal-in-Z phases -> H
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)
    for i in range(N_QUBITS):
        qml.RZ(z[i], wires=i)
    for i in range(N_QUBITS - 1):
        qml.IsingZZ(z[i] * z[i+1], wires=[i, i+1])
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)

def _reupload_feature_map(z, layers=2):
    # Simple re-uploading: repeat (data rotations + entangle)
    for _ in range(layers):
        for i in range(N_QUBITS):
            qml.RY(z[i], wires=i)
            qml.RZ(z[i], wires=i)
        _entangle_chain()

@qml.qnode(qdev)
def quantum_features(z, feature_map="zz"):
    # Start in |+...+>
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)

    # Data feature map (diagonal phases)
    if feature_map == "zz":
        _zz_feature_map(z)
    elif feature_map == "iqp":
        _iqp_feature_map(z)
    elif feature_map == "reupload":
        _reupload_feature_map(z, layers=2)
    else:
        raise ValueError("feature_map must be one of: 'zz', 'iqp', 'reupload'")

    # ✅ Basis change so phase info becomes measurable
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)   # now Z measurement = X in original basis

    # Readout features: local X + nearest-neighbor XX (implemented as Z after H)
    feats = []
    for i in range(N_QUBITS):
        feats.append(qml.expval(qml.PauliZ(i)))  # actually <X_i> before basis change
    for i in range(N_QUBITS - 1):
        feats.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(i+1)))  # actually <X_i X_{i+1}>
    return tuple(feats)

def compute_quantum_feature_matrix(Z, feature_map="zz", angle_scale=np.pi, max_samples=None):
    """
    Z: (N, N_QUBITS), assumed in roughly [-1,1] due to tanh; we scale to angles by angle_scale.
    """
    Z = np.asarray(Z, dtype=np.float64)
    if max_samples is not None:
        Z = Z[:max_samples]

    N = Z.shape[0]
    m = N_QUBITS + (N_QUBITS - 1)  # Z_i plus ZZ_(i,i+1)
    Q = np.zeros((N, m), dtype=np.float64)

    for i in range(N):
        z_angles = angle_scale * Z[i]
        Q[i, :] = np.array(quantum_features(z_angles, feature_map=feature_map), dtype=np.float64)
        if (i + 1) % 500 == 0:
            print(f"  computed quantum features for {i+1}/{N}")

    return Q

# ⚠️ Quantum feature computation can be the slowest part.
# For quick debugging, set MAX_Q_SAMPLES = 2000 or similar.
MAX_Q_SAMPLES = None  # e.g., 2000 for speed, or None for full 10k/10k

Q_train = compute_quantum_feature_matrix(Z_train, feature_map=FEATURE_MAP, max_samples=MAX_Q_SAMPLES)
Q_test  = compute_quantum_feature_matrix(Z_test,  feature_map=FEATURE_MAP, max_samples=MAX_Q_SAMPLES)

print("Quantum feature shapes:", Q_train.shape, Q_test.shape)

  computed quantum features for 500/10000
  computed quantum features for 1000/10000
  computed quantum features for 1500/10000
  computed quantum features for 2000/10000
  computed quantum features for 2500/10000
  computed quantum features for 3000/10000
  computed quantum features for 3500/10000
  computed quantum features for 4000/10000
  computed quantum features for 4500/10000
  computed quantum features for 5000/10000
  computed quantum features for 5500/10000
  computed quantum features for 6000/10000
  computed quantum features for 6500/10000
  computed quantum features for 7000/10000
  computed quantum features for 7500/10000
  computed quantum features for 8000/10000
  computed quantum features for 8500/10000
  computed quantum features for 9000/10000
  computed quantum features for 9500/10000
  computed quantum features for 10000/10000
  computed quantum features for 500/10000
  computed quantum features for 1000/10000
  computed quantum features for 1500/10000
  computed q

In [24]:
# Cell 8 — Hybrid ridge regression on [X, quantum_features(Z)]

# If you truncated quantum features with MAX_Q_SAMPLES, truncate X/y to match.
if MAX_Q_SAMPLES is not None:
    X_train_h = X_train_s[:MAX_Q_SAMPLES]
    y_train_h = y_train[:MAX_Q_SAMPLES]
    X_test_h  = X_test_s[:MAX_Q_SAMPLES]
    y_test_h  = y_test[:MAX_Q_SAMPLES]
else:
    X_train_h, y_train_h = X_train_s, y_train
    X_test_h,  y_test_h  = X_test_s,  y_test

# Augmented features: [X, Q]
Xq_train = np.concatenate([X_train_h, Q_train], axis=1)
Xq_test  = np.concatenate([X_test_h,  Q_test],  axis=1)

# Standardize the augmented matrix too (recommended)
Xq_mu, Xq_sd = standardize_fit(Xq_train)
Xq_train_s = standardize_apply(Xq_train, Xq_mu, Xq_sd)
Xq_test_s  = standardize_apply(Xq_test,  Xq_mu, Xq_sd)

w_hybrid = ridge_fit_closed_form(Xq_train_s, y_train_h, lam=LAMBDA, fit_intercept=True)
pred_hybrid = ridge_predict(Xq_test_s, w_hybrid, fit_intercept=True)

print("Hybrid Ridge ([X, Q(Z)]):", regression_metrics(y_test_h, pred_hybrid))

Hybrid Ridge ([X, Q(Z)]): {'MSE': 3.9732034412279025, 'MAE': 1.5020558105581772, 'Corr': 0.9399708035594666}


In [25]:
# Cell 9 — evaluation summary

results = {}

# Raw baseline
pred_raw = ridge_predict(X_test_s, w_raw, fit_intercept=True)
results["Ridge(raw X)"] = regression_metrics(y_test, pred_raw)

# Compressed baseline
pred_z = ridge_predict(Z_test, w_z, fit_intercept=True)
results["Ridge(compressed Z)"] = regression_metrics(y_test, pred_z)

# Hybrid
results[f"Ridge([X, Q({FEATURE_MAP})])"] = regression_metrics(y_test_h, pred_hybrid)

for k, v in results.items():
    print(f"{k:>22} | MSE={v['MSE']:.6f} | MAE={v['MAE']:.6f} | Corr={v['Corr']:.4f}")

          Ridge(raw X) | MSE=4.571094 | MAE=1.620418 | Corr=0.9306
   Ridge(compressed Z) | MSE=4.571094 | MAE=1.620418 | Corr=0.9306
     Ridge([X, Q(zz)]) | MSE=3.973203 | MAE=1.502056 | Corr=0.9400
